# <b> 02일차 Pandas로 심혈관 CSV 읽기</b>

- 목표: `pandas`로 의료 CSV를 읽고, 구조와 결측치, 기본 통계를 확인합니다.
- 연결: 이 노트북은 메인 실습의 심장병 데이터 파트로 들어가기 전에 보는 짧은 브리지입니다.


In [ ]:
# Shared lecture path bootstrap
from pathlib import Path
from collections import deque
import os
import sys
import unicodedata

try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

SEARCH_ROOTS = (
    Path("/content/drive/MyDrive/Colab Notebooks"),
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
    Path.home(),
)
PROJECT_NAMES = {"의료_이미지_2026", "의료_이미지 2026", "앨리스 - 의료이미지 (26.04)"}

def _norm(text: str) -> str:
    return unicodedata.normalize("NFC", text)

def _is_project_root(path: Path) -> bool:
    return (path / "00_shared" / "utils" / "lecture_paths.py").exists()

def _find_project_root() -> Path:
    override = os.environ.get("PROJECT_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        if _is_project_root(root):
            return root
        raise FileNotFoundError("환경변수 PROJECT_ROOT가 올바른 강의 루트를 가리키지 않습니다.")

    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents]:
        if _is_project_root(root):
            return root

    target_names = {_norm(name) for name in PROJECT_NAMES}
    for base in SEARCH_ROOTS:
        if not base.exists():
            continue
        queue = deque([(base.resolve(), 0)])
        seen = set()
        while queue:
            current, depth = queue.popleft()
            if current in seen:
                continue
            seen.add(current)

            if _norm(current.name) in target_names and _is_project_root(current):
                return current
            if depth >= 4:
                continue

            try:
                children = [
                    child for child in current.iterdir()
                    if child.is_dir() and not child.name.startswith(".") and child.name not in {"__pycache__", ".ipynb_checkpoints"}
                ]
            except OSError:
                continue

            children.sort(key=lambda child: (_norm(child.name) not in target_names, child.name))
            queue.extend((child.resolve(), depth + 1) for child in children)

    raise FileNotFoundError(
        "PROJECT_ROOT를 찾지 못했습니다. Colab에서는 os.environ['PROJECT_ROOT']를 먼저 지정하세요."
    )

def _resolve_day_dir(name: str) -> Path:
    for candidate in {name, unicodedata.normalize("NFC", name), unicodedata.normalize("NFD", name)}:
        path = PROJECT_ROOT / candidate
        if path.exists():
            return path
    return PROJECT_ROOT / name

PROJECT_ROOT = _find_project_root()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
UTILS_ROOT = PROJECT_ROOT / "00_shared" / "utils"
if str(UTILS_ROOT) not in sys.path:
    sys.path.append(str(UTILS_ROOT))

from lecture_paths import DATA_ROOT, MODEL_ROOT, ensure_dir

DAY_DIR = _resolve_day_dir('02일차_의료데이터_탐색_기초실습')
DAY_OUTPUT_ROOT = ensure_dir(DAY_DIR / "99_실행산출물")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"MODEL_ROOT: {MODEL_ROOT}")
print(f"DAY_OUTPUT_ROOT: {DAY_OUTPUT_ROOT}")


## 실행 가이드

- 로컬 실행: bootstrap 셀부터 실행하고, 출력된 `PROJECT_ROOT`, `DATA_ROOT`, `DAY_OUTPUT_ROOT`를 먼저 확인합니다.
- Colab 실행: 01일차 입문 노트북과 같은 방식으로 `PROJECT_ROOT`를 지정한 뒤 bootstrap 셀부터 실행합니다.
- 이 노트북은 `heart_disease.csv`를 기준으로 `pandas`의 최소 문법만 다룹니다.


## AI 도구 활용 체크포인트

- `df.info()`와 `df.isna().sum()` 결과가 무엇을 의미하는지 현재 쓰는 AI 도구에게 설명하게 해보고, 출력과 맞는지 직접 비교합니다.
- 컬럼명을 요약하게 할 때는 반드시 노트북 출력과 대조해 틀린 이름이 없는지 확인합니다.


# Part 1. 데이터 불러오기

## 🎯 학습 목표
- CSV를 DataFrame으로 읽는다.
- 행/열 개수와 주요 컬럼을 확인한다.


In [ ]:
import pandas as pd

csv_path = DATA_ROOT / "heart_disease" / "heart_disease.csv"
df = pd.read_csv(csv_path, keep_default_na=False, na_values=[""])

print(f"csv_path: {csv_path}")
print(f"shape: {df.shape}")

## 1. 첫 몇 행 보기

In [ ]:
df.head()

## 2. 컬럼과 데이터 구조 확인

In [ ]:
print(df.columns.tolist())
print()
df.info()

## 3. 결측치 개수 확인

In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)

# Part 2. 필요한 열만 골라보기

## 🎯 학습 목표
- 자주 보는 열만 따로 뽑는다.
- 조건으로 행을 필터링한다.


In [ ]:
key_cols = ["Age", "Gender", "Blood Pressure", "Cholesterol Level", "BMI", "Heart Disease Status"]
key_df = df[key_cols].copy()
key_df.head()

## 📝 TODO 1

In [ ]:
# 여성 환자 중 심장병이 있는 행만 골라 새로운 DataFrame을 만들어보세요.

# Part 3. 요약 통계와 범주형 분포

## 🎯 학습 목표
- 숫자형 변수의 기본 통계를 읽는다.
- 심장병 유무에 따라 평균 차이를 본다.


In [ ]:
numeric_cols = ["Age", "Blood Pressure", "Cholesterol Level", "BMI", "Sleep Hours"]
df[numeric_cols].describe().T

In [ ]:
df.groupby("Heart Disease Status")[numeric_cols].mean().round(1)

In [ ]:
pd.crosstab(df["Gender"], df["Heart Disease Status"])

## 📝 TODO 2

In [ ]:
# Exercise Habits 컬럼의 분포를 value_counts()로 확인해보세요.

## 의료 해석 질문

- 어떤 수치형 변수의 평균 차이가 심장병 유무와 함께 먼저 보이나요?
- 생활습관 관련 범주형 변수 중 현업에서 추가 확인하고 싶은 항목은 무엇인가요?
